# Challenge 1 — Implement an LRU cache
Question: Design a cache with get(key) and put(key, value) that evicts the least recently used item when its capacity is reached.

Expected approach: Maintain a mapping ordered by recent access. Python's OrderedDict is a straightforward solution.

In [4]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        if capacity <= 0:
            raise ValueError("Capacity must be a positive integer")
        self.capacity = capacity

    def get(self, key: int) -> int:
        if key not in self.cache:
            return None
        else:
            # Move the accessed item to the end to show that it was recently used
            self.cache.move_to_end(key)
            return self.cache[key]

    def put(self, key: int, value: int):
        if key in self.cache:
            # Update the value and move it to the end to show that it was recently used
            self.cache.move_to_end(key)

        self.cache[key] = value

        if len(self.cache) > self.capacity:
            # Remove the first item (the least recently used)
            self.cache.popitem(last=False)

# Example usage:
if __name__ == "__main__":
    cache = LRUCache(2)
    cache.put("A", 100)
    cache.put("B", 200)
    print(cache.get("A"))  # Output: 100
    cache.put("C", 300)    # This will evict key "B"

    print(cache.get("B"))  # Output: None (not found)

100
None


Complexity: Average O(1) for get and put.

Likely follow-ups: What if values can be None? How would you add TTL? Is it thread-safe? How would you make it distributed?

Production answer: For multiple service instances, use Redis if a shared cache is required; an in-memory LRU is local to one process.

# Challenge 2 — Fetch 100 APIs concurrently, but allow only 10 at a time
This is particularly relevant to the job's Python async and enterprise SDK requirements.

In [ ]:
import asyncio
import httpx

async def fetch_all(urls: list[str]):
    semaphore = asyncio.Semaphore(10)  # Limit to 10 concurrent requests

    async with httpx.AsyncClient(timeout=10.0) as client:

        async def fetch(url:str):
            async with semaphore:
                response = await client.get(url)
                response.raise_for_status()  # Raise an error for bad responses
                return response.json()  # Assuming the response is JSON

        tasks = [fetch(url) for url in urls]
        return await asyncio.gather(*tasks)

## What to explain:
- AsyncClient reuses connections.
- Semaphore(10) limits concurrent requests.
- timeout prevents indefinite waits.
- gather() executes I/O operations concurrently and returns results in input order.
- One exception propagates to the caller; other tasks are not automatically cancelled just because gather() raised.
Likely follow-ups: How would you retry HTTP 429 or 503? Preserve partial successes? Cancel outstanding work? Respect a provider-wide quota?

Production improvement: Add bounded retries with jitter, structured logging and a clearly defined partial-failure policy. For very large URL lists, use a fixed-size worker pool rather than create one task per URL.

# Challenge 3 — Implement a retry decorator
Question: Retry transient failures three times with exponential backoff.

In [ ]:
import functools
import random
import time

def retry(max_attempts=3, base_delay=0.5):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except(TimeoutError, ConnectionError):
                    if attempt == max_attempts - 1:
                        raise

                    delay = base_delay * (2 ** attempt)
                    time.sleep(delay + random.uniform(0, 0.1))
        return wrapper
    return decorator

Expected explanation:
"I'd retry only transient failures, use exponential backoff with jitter and limit the number of attempts. I'd respect Retry-After for rate-limited APIs and avoid automatically retrying non-idempotent operations without a safe idempotency mechanism."

Follow-up: Why is time.sleep() unsuitable inside an async function?

It blocks the event loop. An async version should use await asyncio.sleep().

In [ ]:
import httpx

@retry(max_attempts=3, base_delay=0.5)
def fetch_data():
    print("Calling External API")

    response = httpx.get(
        "https://api.example.com/data",
        timeout=5.0
    )

    response.raise_for_status()
    return response.json()

In [ ]:
result = fetch_data()

# Challenge 4 — Implement a rate limiter
Question: Allow a maximum of five requests per user within ten seconds.

Use the sliding-window algorithm.

In [9]:
from collections import defaultdict, deque
from time import monotonic

class RateLimiter:
    def __init__(self, limit=5, window=10):
        self.limit = limit
        self.window = window
        self.requests = defaultdict(deque)

    def allow(self, user_id):
        now = monotonic()
        q = self.requests[user_id]

        while q and q[0] <= now - self.window:
            q.popleft()

        if len(q) >= self.limit:
            return False

        q.append(now)
        return True

Complexity: Amortised O(1) per request because timestamps are appended and removed once.

Production follow-ups: This implementation is process-local, not thread-safe, and doesn't clean up inactive users. For distributed enforcement, consider an atomic Redis implementation using Lua or another server-side atomic operation.

In [12]:
# Create a rate limiter
limiter = RateLimiter(limit=5, window=10)

# Simulate 7 requests from the same user
for i in range(7):
    allowed = limiter.allow("user123")
    print(f"Request {i + 1}: {'Allowed' if allowed else 'Blocked'}")

Request 1: Allowed
Request 2: Allowed
Request 3: Allowed
Request 4: Allowed
Request 5: Allowed
Request 6: Allowed
Request 7: Allowed


In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()
limiter = RateLimiter(limit=5, window=10)

@app.get("/app/data")
def get_data(user_id: str):

    if not limiter.allow(user_id):
        raise HTTPException(
            status_code=429,
            detail="Too many requests"
        )

    return {
        "message": "Request successful",
        "user_id": user_id
    }
